In [0]:
%sql
-- 1. Crear la base de datos para la capa Silver de Salud
CREATE DATABASE IF NOT EXISTS health_fhir_silver;

-- 2. Crear la tabla Delta Silver aplanando los datos cruidos de Bronze
CREATE OR REPLACE TABLE health_fhir_silver.clean_patients_encounters AS
SELECT 
    -- Extracción y limpieza de datos del Paciente
    get_json_object(raw_json, '$.id') AS patient_id,
    get_json_object(raw_json, '$.resourceType') AS resource_type,
    get_json_object(raw_json, '$.gender') AS gender,
    CAST(get_json_object(raw_json, '$.birthDate') AS DATE) AS birth_date,
    
    -- Extracción de campos anidados del objeto 'name' (primer elemento del array [0])
    get_json_object(raw_json, '$.name[0].family') AS family_name,
    get_json_object(raw_json, '$.name[0].given[0]') AS given_name,
    
    -- Extracción de campos anidados del objeto 'encounter'
    get_json_object(raw_json, '$.encounter.id') AS encounter_id,
    get_json_object(raw_json, '$.encounter.class') AS encounter_class,
    CAST(get_json_object(raw_json, '$.encounter.period.start') AS TIMESTAMP) AS encounter_start,
    CAST(get_json_object(raw_json, '$.encounter.period.end') AS TIMESTAMP) AS encounter_end,
    
    -- Extracción del diagnóstico médico dentro del encuentro
    get_json_object(raw_json, '$.encounter.diagnoses[0].code') AS diagnosis_code,
    get_json_object(raw_json, '$.encounter.diagnoses[0].display') AS diagnosis_display,
    
    -- Metadatos de auditoría heredados de Bronze
    _ingestion_timestamp,
    _source_system
FROM health_fhir_bronze.raw_patients_encounters;

In [0]:
%sql
SELECT patient_id, family_name, given_name, gender, encounter_class, diagnosis_display, encounter_start 
FROM health_fhir_silver.clean_patients_encounters;